In [1]:
from pyspark.sql import SparkSession  
import getpass 
username = getpass 
spark = SparkSession.builder.config('spark.ui.port','0').\
        config('spark.sql.warehouse.dir',f"/user/itv022063/warehouse").\
        enableHiveSupport().\
        master('yarn').\
        getOrCreate()

In [2]:
loan_schema="""
       loan_id string,
       total_principle_received float,
total_interest_received float,
total_late_fee_received float,
total_payment_received float,
last_payment_amount float,
last_payment_date string,
next_payment_date string
           """ 

In [3]:
loandf=spark.read.format('csv')\
.schema(loan_schema)\
.option("header","true") \
.load("/user/itv022063/Yash/raw/Loan_Repayment")

In [4]:
loandf.printSchema()

root
 |-- loan_id: string (nullable = true)
 |-- total_principle_received: float (nullable = true)
 |-- total_interest_received: float (nullable = true)
 |-- total_late_fee_received: float (nullable = true)
 |-- total_payment_received: float (nullable = true)
 |-- last_payment_amount: float (nullable = true)
 |-- last_payment_date: string (nullable = true)
 |-- next_payment_date: string (nullable = true)



In [5]:
from pyspark.sql.functions import *
loandf=loandf.withColumn("injest_date",current_timestamp())

In [6]:
columns_to_check = ["total_principle_received", "total_interest_received",
"total_late_fee_received", "total_payment_received", "last_payment_amount"]

In [7]:
loandf=loandf.na.drop(subset = columns_to_check)

In [8]:
display(loandf)

loan_id,total_principle_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,injest_date
68407277,3600.0,821.72,0.0,4421.724,122.67,Jan-2019,null,2025-12-31 14:07:...
68355089,24700.0,979.66,0.0,25679.66,926.35,Jun-2016,null,2025-12-31 14:07:...
68341763,20000.0,2705.92,0.0,22705.924,15813.3,Jun-2017,null,2025-12-31 14:07:...
66310712,19102.35,12361.66,0.0,31464.01,829.9,Feb-2019,Apr-2019,2025-12-31 14:07:...
68476807,10400.0,1340.5,0.0,11740.5,10128.96,Jul-2016,null,2025-12-31 14:07:...
68426831,11950.0,1758.95,0.0,13708.948,7653.56,May-2017,null,2025-12-31 14:07:...
68476668,20000.0,1393.8,0.0,21393.8,15681.05,Nov-2016,null,2025-12-31 14:07:...
67275481,20000.0,1538.51,0.0,21538.51,14618.23,Jan-2017,null,2025-12-31 14:07:...
68466926,10000.0,998.97,0.0,10998.972,1814.48,Aug-2018,null,2025-12-31 14:07:...
68616873,8000.0,939.58,0.0,8939.58,4996.24,Apr-2017,null,2025-12-31 14:07:...


In [9]:
loan = (
    loandf
    .withColumn(
        "total_payment_received",
        when(
            (col("total_payment_received") == 0.0) &
            (col("total_principle_received") != 0.0),
            col("total_principle_received")
            + col("total_interest_received")
            + col("total_late_fee_received")
        ).otherwise(col("total_payment_received"))
    )
)

In [10]:
loan=loan.filter("total_payment_received != 0.0")

In [11]:
loan=(
    loan
    .withColumn(
        "last_payment_date", 
        when(
            (col("last_payment_date") == 0.0),
            None
        ).otherwise(col("last_payment_date")
      )
    )
)

In [12]:
loan_repayment=(
    loan
    .withColumn("next_payment_date", 
                when((col("next_payment_date") == 0.0),
                     None
                    ).otherwise(col("next_payment_date")
                               )
               )
)

In [13]:
loan_repayment

loan_id,total_principle_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,injest_date
68407277,3600.0,821.72,0.0,4421.724,122.67,Jan-2019,null,2025-12-31 14:07:...
68355089,24700.0,979.66,0.0,25679.66,926.35,Jun-2016,null,2025-12-31 14:07:...
68341763,20000.0,2705.92,0.0,22705.924,15813.3,Jun-2017,null,2025-12-31 14:07:...
66310712,19102.35,12361.66,0.0,31464.01,829.9,Feb-2019,Apr-2019,2025-12-31 14:07:...
68476807,10400.0,1340.5,0.0,11740.5,10128.96,Jul-2016,null,2025-12-31 14:07:...
68426831,11950.0,1758.95,0.0,13708.948,7653.56,May-2017,null,2025-12-31 14:07:...
68476668,20000.0,1393.8,0.0,21393.8,15681.05,Nov-2016,null,2025-12-31 14:07:...
67275481,20000.0,1538.51,0.0,21538.51,14618.23,Jan-2017,null,2025-12-31 14:07:...
68466926,10000.0,998.97,0.0,10998.972,1814.48,Aug-2018,null,2025-12-31 14:07:...
68616873,8000.0,939.58,0.0,8939.58,4996.24,Apr-2017,null,2025-12-31 14:07:...


In [14]:
loan_repayment.write \
.format("parquet") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loan_repayment_parquet") \
.save()

In [15]:
loan_repayment.write \
.format("csv") \
.mode("overwrite") \
.option("path", "/user/itv022063/Yash/raw/loan_repayment_csv") \
.save()

In [16]:
spark.read.format("parquet").load("/user/itv022063/Yash/raw/loan_repayment_parquet")

loan_id,total_principle_received,total_interest_received,total_late_fee_received,total_payment_received,last_payment_amount,last_payment_date,next_payment_date,injest_date
68407277,3600.0,821.72,0.0,4421.724,122.67,Jan-2019,null,2025-12-31 14:07:...
68355089,24700.0,979.66,0.0,25679.66,926.35,Jun-2016,null,2025-12-31 14:07:...
68341763,20000.0,2705.92,0.0,22705.924,15813.3,Jun-2017,null,2025-12-31 14:07:...
66310712,19102.35,12361.66,0.0,31464.01,829.9,Feb-2019,Apr-2019,2025-12-31 14:07:...
68476807,10400.0,1340.5,0.0,11740.5,10128.96,Jul-2016,null,2025-12-31 14:07:...
68426831,11950.0,1758.95,0.0,13708.948,7653.56,May-2017,null,2025-12-31 14:07:...
68476668,20000.0,1393.8,0.0,21393.8,15681.05,Nov-2016,null,2025-12-31 14:07:...
67275481,20000.0,1538.51,0.0,21538.51,14618.23,Jan-2017,null,2025-12-31 14:07:...
68466926,10000.0,998.97,0.0,10998.972,1814.48,Aug-2018,null,2025-12-31 14:07:...
68616873,8000.0,939.58,0.0,8939.58,4996.24,Apr-2017,null,2025-12-31 14:07:...
